<a href="https://colab.research.google.com/github/piyakorn-h/Super-AI-Engineer-Season-6/blob/main/thai_word_segmentation_wangchanberta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🇹🇭 Thai Word Segmentation with WangchanBERTa

**Task**: Character-level sequence labeling (BIE tagging)
- `B_WORD` = Beginning of a word
- `I_WORD` = Inside a word
- `E_WORD` = End of a word

**Model**: `airesearch/wangchanberta-base-att-spm-uncased`

**Pipeline**:
1. Install dependencies
2. Load & preprocess data
3. Tokenize with character-level alignment
4. Fine-tune WangchanBERTa (Token Classification)
5. Predict on test set
6. Export `ws_sample_submission.csv`

## 1. Install Dependencies

In [ ]:
!pip install transformers datasets seqeval sentencepiece -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00


## 2. Mount Google Drive (วางไฟล์ train ใน Drive แล้ว mount)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ===== กำหนด path ของไฟล์ต่าง ๆ =====
# แก้ path ด้านล่างให้ตรงกับที่วางไฟล์ไว้ใน Drive
TRAIN_DIR   = '/content/drive/MyDrive/hackathon_ws/train'  # โฟลเดอร์ train
TEST_FILE   = '/content/drive/MyDrive/hackathon_ws/ws_test.txt'
SAMPLE_SUB  = '/content/drive/MyDrive/hackathon_ws/ws_sample_submission.csv'
OUTPUT_SUB  = '/content/drive/MyDrive/hackathon_ws/submission.csv'

## 3. Imports & Config

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from datasets import Dataset as HFDataset
from seqeval.metrics import classification_report

# ===== Labels =====
LABEL_LIST  = ['B_WORD', 'E_WORD', 'I_WORD']   # sorted alphabetically for consistency
LABEL2ID    = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL    = {i: l for l, i in LABEL2ID.items()}

MODEL_NAME  = 'airesearch/wangchanberta-base-att-spm-uncased'
MAX_LEN     = 416    # max subword tokens (leave room for [CLS]/[SEP])
STRIDE      = 128    # overlap window for long texts
BATCH_SIZE  = 8
EPOCHS      = 5
LR          = 2e-5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 4. Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## 5. Data Loading Helpers

**สมมติรูปแบบ train data**: แต่ละไฟล์ใน `TRAIN_DIR` เป็น plain text ที่คั่นคำด้วย `|` เช่น `ที่|ยัง|สถาน|การณ์|...`  
ถ้า format ต่างกัน ให้แก้ `parse_train_file()` ด้านล่าง

In [ ]:
def word_to_bie(word: str):
    """Convert a word string to a list of BIE labels for each character."""
    chars = list(word)
    n = len(chars)
    if n == 1:
        return chars, ['B_WORD']   # single-char word: treat as B (or could be B+E; adjust per competition spec)
    labels = ['B_WORD'] + ['I_WORD'] * (n - 2) + ['E_WORD']
    return chars, labels


def parse_train_file(filepath: str):
    """
    Parse one training file.
    Expected format: words separated by '|' (pipe) on each line.
    Returns: list of (chars, labels) sentence pairs.

    *** แก้ฟังก์ชันนี้ถ้า format ของ train data ต่างออกไป ***
    """
    sentences = []  # list of (char_list, label_list)
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            words = line.split('|')
            chars_all, labels_all = [], []
            for word in words:
                if not word:
                    continue
                c, l = word_to_bie(word)
                chars_all.extend(c)
                labels_all.extend(l)
            if chars_all:
                sentences.append((chars_all, labels_all))
    return sentences


def load_train_data(train_dir: str):
    all_sentences = []
    files = glob.glob(os.path.join(train_dir, '**', '*.txt'), recursive=True)
    print(f'Found {len(files)} training files')
    for f in files:
        all_sentences.extend(parse_train_file(f))
    print(f'Total training sentences: {len(all_sentences)}')
    return all_sentences


# Load training data
train_sentences = load_train_data(TRAIN_DIR)

## 6. Tokenize with Character Alignment

WangchanBERTa ใช้ SentencePiece ซึ่ง tokenize เป็น subwords  
เราต้อง align character-level labels → subword-level labels (ใช้ label ของ first subword เท่านั้น, subwords ถัดไป mask ด้วย -100)

In [ ]:
def tokenize_and_align(sentences, tokenizer, max_len=MAX_LEN, stride=STRIDE, label2id=LABEL2ID):
    """
    sentences: list of (char_list, label_list)
    Returns HuggingFace Dataset
    """
    all_input_ids, all_attention_masks, all_labels = [], [], []

    for chars, char_labels in sentences:
        text = ''.join(chars)
        # Tokenize with offset mapping so we can align chars → tokens
        encoding = tokenizer(
            text,
            return_offsets_mapping=True,
            truncation=True,
            max_length=max_len,
            stride=stride,
            return_overflowing_tokens=True,
            padding='max_length',
        )

        for chunk_idx in range(len(encoding['input_ids'])):
            input_ids       = encoding['input_ids'][chunk_idx]
            attention_mask  = encoding['attention_mask'][chunk_idx]
            offset_mapping  = encoding['offset_mapping'][chunk_idx]

            token_labels = []
            prev_char_end = None
            for offset in offset_mapping:
                start, end = offset
                if start == 0 and end == 0:
                    # Special tokens [CLS], [SEP], [PAD]
                    token_labels.append(-100)
                elif start == prev_char_end:
                    # Continuation subword → mask
                    token_labels.append(-100)
                else:
                    # First subword of a character span → assign label of the first char in span
                    char_idx = start  # char index in original string
                    if char_idx < len(char_labels):
                        token_labels.append(label2id[char_labels[char_idx]])
                    else:
                        token_labels.append(-100)
                prev_char_end = end

            all_input_ids.append(input_ids)
            all_attention_masks.append(attention_mask)
            all_labels.append(token_labels)

    dataset = HFDataset.from_dict({
        'input_ids':      all_input_ids,
        'attention_mask': all_attention_masks,
        'labels':         all_labels,
    })
    dataset.set_format('torch')
    return dataset


print('Tokenizing training data...')
train_dataset = tokenize_and_align(train_sentences, tokenizer)
print(f'Train chunks: {len(train_dataset)}')

## 7. Load Model

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
model.to(device)

## 8. Training

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    true_preds, true_labels = [], []
    for pred_row, label_row in zip(predictions, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                p_seq.append(ID2LABEL[p])
                l_seq.append(ID2LABEL[l])
        true_preds.append(p_seq)
        true_labels.append(l_seq)
    report = classification_report(true_labels, true_preds, output_dict=True, zero_division=0)
    return {
        'precision': report['weighted avg']['precision'],
        'recall':    report['weighted avg']['recall'],
        'f1':        report['weighted avg']['f1-score'],
    }


training_args = TrainingArguments(
    output_dir              = './ws_checkpoints',
    evaluation_strategy     = 'no',       # เปลี่ยนเป็น 'epoch' ถ้ามี validation set
    save_strategy           = 'epoch',
    learning_rate           = LR,
    per_device_train_batch_size = BATCH_SIZE,
    num_train_epochs        = EPOCHS,
    weight_decay            = 0.01,
    warmup_ratio            = 0.1,
    load_best_model_at_end  = False,
    fp16                    = torch.cuda.is_available(),
    logging_steps           = 50,
    report_to               = 'none',
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)

trainer.train()

## 9. Inference on Test Data

In [ ]:
def predict_text(text: str, model, tokenizer, max_len=MAX_LEN, stride=STRIDE):
    """
    Predict BIE label for every character in `text`.
    Handles long texts via sliding window; overlapping predictions are averaged.
    Returns: list of label strings, one per character.
    """
    model.eval()
    chars = list(text)
    n_chars = len(chars)

    # Accumulate logits per character
    char_logits = np.zeros((n_chars, len(LABEL_LIST)), dtype=np.float32)
    char_counts = np.zeros(n_chars, dtype=np.int32)

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        padding='max_length',
        return_tensors='pt',
    )

    n_chunks = encoding['input_ids'].shape[0]
    for chunk_idx in range(n_chunks):
        input_ids      = encoding['input_ids'][chunk_idx].unsqueeze(0).to(device)
        attention_mask = encoding['attention_mask'][chunk_idx].unsqueeze(0).to(device)
        offset_mapping = encoding['offset_mapping'][chunk_idx]

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits[0].cpu().numpy()  # (seq_len, num_labels)

        prev_end = None
        for token_idx, (start, end) in enumerate(offset_mapping.tolist()):
            if start == 0 and end == 0:
                prev_end = None
                continue
            # Map each character in [start, end) to this token's logits
            for char_idx in range(start, min(end, n_chars)):
                char_logits[char_idx] += logits[token_idx]
                char_counts[char_idx] += 1
            prev_end = end

    # Average logits and argmax
    char_counts = np.maximum(char_counts, 1)  # avoid div by zero
    avg_logits = char_logits / char_counts[:, None]
    pred_ids   = np.argmax(avg_logits, axis=-1)
    pred_labels = [ID2LABEL[p] for p in pred_ids]
    return pred_labels


# Read test text
with open(TEST_FILE, encoding='utf-8') as f:
    test_text = f.read()

print(f'Test text length: {len(test_text)} characters')
print('Running inference...')
pred_labels = predict_text(test_text, model, tokenizer)
print('Done!')
print(f'Predicted {len(pred_labels)} labels')

## 10. Write Submission CSV

In [ ]:
import pandas as pd

# Build a character-index → label dict (0-indexed)
label_map = {i: lbl for i, lbl in enumerate(pred_labels)}

# Load sample submission to get exact Id list
sample_df = pd.read_csv(SAMPLE_SUB)

def fill_prediction(row_id):
    # Submission Ids are 1-indexed based on the test file characters
    char_idx = row_id - 1  # convert to 0-indexed
    return label_map.get(char_idx, 'B_WORD')  # fallback to B_WORD if missing

sample_df['Predicted'] = sample_df['Id'].apply(fill_prediction)

sample_df.to_csv(OUTPUT_SUB, index=False)
print(f'Submission saved to: {OUTPUT_SUB}')
sample_df.head(10)

## 11. (Optional) Download submission locally

In [ ]:
from google.colab import files
files.download(OUTPUT_SUB)

---
## 📝 Notes & Tips

### รูปแบบ Train Data
ถ้าโฟลเดอร์ train มี format ต่างจาก `word|word|word` ให้แก้ฟังก์ชัน `parse_train_file()` ในเซลล์ที่ 5

### Single-character words
ปัจจุบัน single-char word ถูก label เป็น `B_WORD` คนเดียว ตรวจสอบ spec ของ competition ว่า single-char ควรเป็น `B_WORD`, `E_WORD`, หรือทั้งคู่

### Hyperparameter Tuning
- เพิ่ม `EPOCHS` เป็น 10 ถ้ามีเวลา
- ลอง `LR = 3e-5` หรือ `1e-5`
- เพิ่ม `MAX_LEN = 512` ถ้า GPU memory พอ

### Validation Split
แนะนำให้แบ่ง 10% ของ train มาเป็น validation แล้วเปลี่ยน `evaluation_strategy='epoch'` และ `load_best_model_at_end=True`